# Partie 1 : CIFAR

In [ ]:
import tensorflow as tf

# 1. Chargement du dataset CIFAR-10
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
(x_train_no_norm, y_train), (x_test_no_norm, y_test) = tf.keras.datasets.cifar10.load_data()
# 2. Normalisation des images (Fortement recommandé)
# Les pixels ont des valeurs entre 0 et 255. On les divise par 255.0 
# pour que le réseau de neurones apprenne plus facilement avec des valeurs entre 0 et 1.
x_train, x_test = x_train / 255.0, x_test / 255.0

# Affichage des dimensions pour vérifier
print("Format des données d'entraînement :", x_train.shape)
print("Format des étiquettes d'entraînement :", y_train.shape)

## Analyse exploratoire des données

### Distribution des classes

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
import numpy as np

# On définit les noms des classes pour que le graphique soit lisible
noms_classes = ['Avion', 'Automobile', 'Oiseau', 'Chat', 'Cerf', 
                'Chien', 'Grenouille', 'Cheval', 'Bateau', 'Camion']

# On compte le nombre d'occurrences de chaque classe dans y_train
# y_train contient les étiquettes de 0 à 9. np.unique compte combien de fois chaque chiffre apparaît.
classes, nombres = np.unique(y_train, return_counts=True)

# Création du graphique en barres
plt.figure(figsize=(10, 6))
plt.bar(noms_classes, nombres, color='skyblue', edgecolor='black')

# Personnalisation (titre, légendes)
plt.title("Distribution des classes dans CIFAR-10 (Données d'entraînement)", fontsize=14)
plt.xlabel("Classes", fontsize=12)
plt.ylabel("Nombre d'images", fontsize=12)
plt.xticks(rotation=45) # On incline le texte pour éviter que les mots se chevauchent

# On ajoute le nombre exact au-dessus de chaque barre pour plus de clarté
for i in range(len(classes)):
    plt.text(i, nombres[i] + 100, str(nombres[i]), ha='center')

# Affichage
plt.tight_layout()
plt.show()

### Affichage des 10 premières images

In [ ]:
plt.figure(figsize=(12, 5))

# On affiche les 10 premières images du dataset
for i in range(10):
    plt.subplot(2, 5, i + 1)
    # x_train[i] est déjà normalisé entre 0 et 1, imshow l'affiche sans problème
    plt.imshow(x_train[i]) 
    
    # On récupère le nom de la classe correspondante
    index_classe = int(y_train[i][0])
    plt.title(noms_classes[index_classe])
    
    plt.axis('off') # On cache les axes (les numéros de pixels) pour faire plus joli

plt.tight_layout()
plt.show()

## Algortihme de ML 

### XGBoost

In [ ]:
x_train_xgb = x_train.reshape(50000,-1)
x_test_xgb = x_test.reshape(10000,-1)


In [ ]:
from sklearn.model_selection import GridSearchCV
import xgboost as xgb

model = xgb.XGBClassifier(device='cuda')

param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.1, 0.01],
    'n_estimators': [100, 200]
}

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5)
grid_search.fit(x_train_xgb, y_train)



print(f"Meilleurs paramètres : {grid_search.best_params_}")

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split



model_xgb = xgb.XGBClassifier(learning_rate=0.1, max_depth=7, n_estimators=200)
model_xgb.fit(x_train_xgb,y_train)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
y_pred_xgb = model_xgb.predict(x_test_xgb)
print(f'Accuracy : {accuracy_score(y_test,y_pred_xgb)}')
print(f'report : {classification_report(y_test, y_pred_xgb)}')

class_names = ["Chat", "Chien", "Oiseau"] # Remplacez par vos vrais noms dans l'ordre

# 2. Calculer la matrice
cm = confusion_matrix(y_test, y_pred_xgb)

# 3. Afficher la matrice avec les noms
fig, ax = plt.subplots(figsize=(8, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=noms_classes)

# Tracer avec une palette de couleurs (cmap)
disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation='vertical')

plt.title("Matrice de Confusion")
plt.show()

In [ ]:
plt.imshow(x_test[1].reshape(32, 32, 3)) # Remettre en forme pour l'affichage
plt.title(f"Vrai: {y_test[1]}, Prédit: {y_pred_xgb[1]}")
plt.show()

## RandomForest avec PCA

In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

x_train_flat = x_train.reshape(50000, -1)
# On crée une chaîne de traitement (Pipeline)
model_rf_pca = Pipeline([
    ('compression_pca', PCA(n_components=100)), # Étape 1 : Réduire à 100 composantes
    ('classifieur_rf', RandomForestClassifier()) # Étape 2 : Le Random Forest
])

param_grid = {
    'classifieur_rf__max_depth': [10, 15, 20], 
    'classifieur_rf__n_estimators': [100, 200]
}

grid_search = GridSearchCV(estimator=model_rf_pca, param_grid=param_grid, cv=5)
grid_search.fit(x_train_flat, y_train.ravel())


print(f"Meilleurs paramètres : {grid_search.best_params_}")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

x_train_flat = x_train.reshape(50000, -1)
# On crée une chaîne de traitement (Pipeline)
model_rf_pca = Pipeline([
    ('compression_pca', PCA(n_components=100)), # Étape 1 : Réduire à 100 composantes
    ('classifieur_rf', RandomForestClassifier()) # Étape 2 : Le Random Forest
])

model_rf_pca.fit(x_train_flat, y_train.ravel())

# CNN classique

## Optimisation avec Keras Tuner

In [ ]:
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# 1. Définition de la fonction de construction du modèle
def build_model(hp):
    model = Sequential()
    
    # --- PREMIÈRE COUCHE CONVOLUTIVE ---
    hp_filters_1 = hp.Int('conv_1_filters', min_value=32, max_value=128, step=32) # choix du nombre de filtres
    model.add(Conv2D(filters=hp_filters_1, kernel_size=(3, 3), activation='relu', input_shape=(32, 32, 3)))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    
    # --- DEUXIÈME COUCHE CONVOLUTIVE ---
    hp_filters_2 = hp.Int('conv_2_filters', min_value=64, max_value=256, step=64)
    model.add(Conv2D(filters=hp_filters_2, kernel_size=(3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    
    # --- TROISIÈME COUCHE CONVOLUTIVE (NOUVELLE) ---
    hp_filters_3 = hp.Int('conv_3_filters', min_value=64, max_value=256, step=64)
    model.add(Conv2D(filters=hp_filters_3, kernel_size=(3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    
    # --- PASSAGE EN 1D (FLATTEN) ---
    model.add(Flatten())
    
    # --- COUCHE CACHÉE (DENSE) ---
    hp_units = hp.Int('dense_units', min_value=128, max_value=512, step=128)
    model.add(Dense(units=hp_units, activation='relu'))
    
    # --- DROPOUT (Contre le surapprentissage) ---
    hp_dropout = hp.Float('dropout_rate', min_value=0.2, max_value=0.5, step=0.1)
    model.add(Dropout(rate=hp_dropout))
    
    # --- COUCHE DE SORTIE (CIFAR-10 = 10 classes) ---
    model.add(Dense(10, activation='softmax'))
    
    # --- COMPILATION ---
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    
    return model

# 2. Instanciation du Tuner
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy', 
    max_epochs=10,            
    factor=3,
    directory='mon_dossier_tuner', 
    project_name='optimisation_cifar10_3couches' 
)

# 3. Création d'un callback d'arrêt anticipé
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)

# 4. Lancement de la recherche
print("Lancement de la recherche des meilleurs hyperparamètres avec 3 couches...")
tuner.search(x_train, y_train, 
             epochs=20, 
             validation_split=0.2, 
             callbacks=[stop_early])

# 5. Récupération des résultats
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\n--- RECHERCHE TERMINÉE ---")
print(f"Meilleur nombre de filtres (Conv 1) : {best_hps.get('conv_1_filters')}")
print(f"Meilleur nombre de filtres (Conv 2) : {best_hps.get('conv_2_filters')}")
print(f"Meilleur nombre de filtres (Conv 3) : {best_hps.get('conv_3_filters')}") # Affichage de la 3ème couche
print(f"Meilleurs neurones (Dense) : {best_hps.get('dense_units')}")
print(f"Meilleur taux de Dropout : {best_hps.get('dropout_rate')}")
print(f"Meilleur Learning Rate : {best_hps.get('learning_rate')}")

## Construction du modèle

In [ ]:
# 1. On demande au Tuner de reconstruire le modèle avec les MEILLEURS paramètres trouvés
meilleurs_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
modele_champion = tuner.hypermodel.build(meilleurs_hps)

# 2. Configuration de l'arrêt anticipé (Early Stopping) revisité
stop_early_final = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=5,                 # On est plus tolérant (on attend 5 époques sans amélioration)
    restore_best_weights=True   # CRUCIAL : à la fin, on recharge les poids de la meilleure époque, pas ceux de la dernière !
)

# 3. Lancement de l'entraînement final
print("Début de l'entraînement du modèle champion...")

# On sauvegarde l'historique pour pouvoir tracer des graphiques ensuite
historique = modele_champion.fit(
    x_train, y_train,
    epochs=50,                  # On met un grand nombre d'époques, l'Early Stopping l'arrêtera au bon moment
    batch_size=64,              # Le modèle analyse les images par paquets de 64 (plus rapide et plus stable)
    validation_split=0.2,       # On garde 20% de x_train pour valider à chaque époque
    callbacks=[stop_early_final]
)

# 4. Le verdict final sur les données de TEST (x_test, y_test)
print("\n--- ÉVALUATION FINALE ---")
perte, precision = modele_champion.evaluate(x_test, y_test)
print(f"Précision finale sur les données de test inconnues : {precision * 100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

# --- GRAPHIQUE 1 : LA PRÉCISION (ACCURACY) ---
plt.subplot(1, 2, 1)
plt.plot(historique.history['accuracy'], label='Entraînement', color='blue', linewidth=2)
plt.plot(historique.history['val_accuracy'], label='Validation', color='orange', linewidth=2)
plt.title('Évolution de la Précision', fontsize=14)
plt.xlabel('Époques', fontsize=12)
plt.ylabel('Précision', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

# --- GRAPHIQUE 2 : L'ERREUR (LOSS) ---
plt.subplot(1, 2, 2)
plt.plot(historique.history['loss'], label='Entraînement', color='blue', linewidth=2)
plt.plot(historique.history['val_loss'], label='Validation', color='orange', linewidth=2)
plt.title('Évolution de l\'Erreur', fontsize=14)
plt.xlabel('Époques', fontsize=12)
plt.ylabel('Erreur (Loss)', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# Finetuning - EfficientNet b4

## Optimisation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision.transforms import v2
import optuna

val_split = 0.2
batch_size = 64
img_size = 224

x_tensor = torch.tensor(x_train_no_norm, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
y_tensor = torch.tensor(y_train, dtype=torch.long).squeeze()

full_ds = TensorDataset(x_tensor, y_tensor)
len_val = int(val_split * len(full_ds))
len_train = len(full_ds) - len_val
train_ds, val_ds = random_split(full_ds, [len_train, len_val])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device utilisé : {device}")

train_transforms = v2.Compose([
    v2.ToDtype(torch.float32, scale=False),  
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=15),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = v2.Compose([
    v2.ToDtype(torch.float32, scale=False),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


def objective(trial):
    print(f"\nDémarrage du Trial #{trial.number}")

    lr            = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    dropout_rate  = trial.suggest_float("dropout_rate", 0.2, 0.6)
    weight_decay  = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)

    base_model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
    base_model.classifier = nn.Sequential(
        nn.Dropout(p=dropout_rate),
        nn.Linear(in_features=1792, out_features=10)
    )

    model = nn.Sequential(
        nn.Upsample(size=(img_size, img_size), mode='bilinear', align_corners=False),
        base_model
    ).to(device)

    for param in model.parameters():
        param.requires_grad = False
    for param in model[1].features[-1].parameters():
        param.requires_grad = True
    for param in model[1].classifier.parameters():
        param.requires_grad = True

    model.eval()
    model[1].classifier.train()

    parametres_a_entrainer = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(parametres_a_entrainer, lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    epochs = 5

    for epoch in range(epochs):
        model[1].classifier.train()

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            inputs = train_transforms(inputs)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        model.eval()
        correct = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                # application des val_transforms (normalisation)
                inputs = val_transforms(inputs)

                outputs = model(inputs)
                correct += (outputs.argmax(1) == labels).sum().item()

        val_acc = correct / len(val_ds)

        trial.report(val_acc, epoch)
        if trial.should_prune():
            del model
            torch.cuda.empty_cache()
            raise optuna.exceptions.TrialPruned()

    del model
    torch.cuda.empty_cache()
    return val_acc


if __name__ == "__main__":
    print("\nDémarrage de l'optimisation")
    study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=20)

    print("\nOPTIMISATION TERMINÉE")
    print("Meilleur score de validation :", study.best_value)
    print("Meilleurs Hyperparamètres trouvés :")
    for key, value in study.best_params.items():
        print(f"  - {key} : {value}")

## Phase 1

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision.transforms import v2


val_split = 0.2
batch_size = 64
img_size = 224

x_tensor = torch.tensor(x_train_no_norm, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
y_tensor = torch.tensor(y_train, dtype=torch.long).squeeze()

full_ds = TensorDataset(x_tensor, y_tensor)
len_val = int(val_split * len(full_ds))
len_train = len(full_ds) - len_val
train_ds, val_ds = random_split(full_ds, [len_train, len_val])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

base_model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
base_model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features=1792, out_features=10)
)

model = nn.Sequential(
    nn.Upsample(size=(img_size, img_size), mode='bilinear', align_corners=False),
    base_model
)

for param in model.parameters():
    param.requires_grad = False
for param in model[1].features[-1].parameters():
    param.requires_grad = True
for param in model[1].classifier.parameters():
    param.requires_grad = True

model = model.to(device)
model.eval()
model[1].classifier.train()

parametres_a_entrainer = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(parametres_a_entrainer, lr=1e-3)
criterion = nn.CrossEntropyLoss()

train_transforms = v2.Compose([
    v2.ToDtype(torch.float32, scale=False),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=15),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = v2.Compose([
    v2.ToDtype(torch.float32, scale=False),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("\nLANCEMENT DE L'ENTRAÎNEMENT")

best_loss = float('inf')
patience, patience_counter = 3, 0

for epoch in range(10):
    model[1].classifier.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        inputs = train_transforms(inputs) 

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    val_loss = 0.0
    correct = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            inputs = val_transforms(inputs)  
            outputs = model(inputs)
            val_loss += criterion(outputs, labels).item()
            correct += (outputs.argmax(1) == labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = correct / len(val_ds)
    print(f"Époque {epoch+1}/10 - Loss train: {running_loss/len(train_loader):.4f} "
          f"- Loss val: {val_loss:.4f} - Acc val: {val_acc:.4f}")

    if val_loss < best_loss:
        best_loss = val_loss
        patience_counter = 0  
        torch.save(model[1].state_dict(), 'efficientnet_phase1.pth')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping !")
            break

print("\nPhase terminée ! Meilleurs poids sauvegardés.")

## Phase 2

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import efficientnet_b4
from torchvision.transforms import v2

print("Reconstruction et chargement des poids précédents.")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
img_size = 224

base_model = efficientnet_b4(weights=None)
base_model.classifier = nn.Sequential(    
    nn.Dropout(p=0.4),
    nn.Linear(in_features=1792, out_features=10)
)

model = nn.Sequential(
    nn.Upsample(size=(img_size, img_size), mode='bilinear', align_corners=False),
    base_model
).to(device)

model[1].load_state_dict(torch.load('efficientnet_phase1.pth'))
print("Poids de la Phase 1 chargés.")

for param in model.parameters():
    param.requires_grad = True

for module in model.modules():
    if isinstance(module, nn.BatchNorm2d):
        module.eval()
        for param in module.parameters():
            param.requires_grad = False

parametres_a_entrainer = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(parametres_a_entrainer, lr=1e-5)
criterion = nn.CrossEntropyLoss()

# FIX : v2 au lieu de T, + ToDtype + Normalize
train_transforms = v2.Compose([
    v2.ToDtype(torch.float32, scale=False),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=15),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = v2.Compose([
    v2.ToDtype(torch.float32, scale=False),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

best_loss = float('inf')
patience, patience_counter = 4, 0
epochs = 15

print("\nLANCEMENT DE L'ENTRAÎNEMENT")

for epoch in range(epochs):
    model.train()
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        inputs = train_transforms(inputs)  # FIX : train_transforms au lieu de data_augmentation

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    val_loss = 0.0
    correct = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            inputs = val_transforms(inputs)  # FIX : normalisation manquante en validation
            outputs = model(inputs)
            val_loss += criterion(outputs, labels).item()
            correct += (outputs.argmax(1) == labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = correct / len(val_ds)
    print(f"Époque {epoch+1}/{epochs} - Loss train: {running_loss/len(train_loader):.4f} "
          f"- Loss val: {val_loss:.4f} - Acc val: {val_acc:.4f}")

    if val_loss < best_loss:
        best_loss = val_loss
        patience_counter = 0  # FIX : reset manquant
        torch.save(model[1].state_dict(), 'efficientnet_final.pth')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping déclenché.")
            break

print("\nTerminé. Modèle sauvegardé.")

## Test

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b4
from torchvision.transforms import v2
from torch.utils.data import DataLoader, TensorDataset

print("PRÉPARATION DES DONNÉES DE TEST")
x_test_tensor = torch.tensor(x_test_no_norm, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
y_test_tensor = torch.tensor(y_test, dtype=torch.long).squeeze()

test_ds = TensorDataset(x_test_tensor, y_test_tensor)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

print("=== RECONSTRUCTION ET CHARGEMENT DU MODÈLE ===")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_model = efficientnet_b4(weights=None)
base_model.classifier = nn.Sequential(    
    nn.Dropout(p=0.4),
    nn.Linear(in_features=1792, out_features=10)
)

model = nn.Sequential(
    nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False),
    base_model
).to(device)

model[1].load_state_dict(torch.load('efficientnet_final.pth', map_location=device))
print("Poids rechargés avec succès !")

# FIX : normalisation du jeu de test
test_transforms = v2.Compose([
    v2.ToDtype(torch.float32, scale=False),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("\n=== LANCEMENT DES PRÉDICTIONS ===")
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        inputs = test_transforms(inputs)  # FIX : normalisation manquante
        outputs = model(inputs)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"\nRÉSULTAT FINAL : Précision sur l'ensemble de test = {accuracy * 100:.2f}%")